# 02 — Modèle de référence : TF-IDF + régression logistique
**Objectif :** entraîner un premier modèle et mesurer ses performances sur la validation.

**Prérequis :** exécuter le notebook 01. **Sorties :** modèle, métriques et prédictions sauvegardés pour le notebook 03.

TF-IDF représente les messages par leurs mots et groupes de deux mots. La régression logistique prédit ensuite la catégorie. Les réglages de ton code sont conservés.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Ouvrir Jupyter depuis le dossier projet_banking77.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
for directory in (DATA_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

from time import perf_counter
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

## 1. Recharger le même découpage

In [ ]:
required = [DATA_DIR / name for name in ("train.csv", "validation.csv", "categories.json")]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}. Exécuter d'abord le notebook 01.")
train_df = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
val_df = pd.read_csv(DATA_DIR / "validation.csv", keep_default_na=False)
categories = pd.Index(json.loads((DATA_DIR / "categories.json").read_text(encoding="utf-8")))
print(f"Train : {len(train_df)} | Validation : {len(val_df)} | Catégories : {len(categories)}")

## 2. Entraîner le modèle

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=42,
    )),
])

start = perf_counter()
baseline.fit(train_df["text"], train_df["category"])
training_seconds = perf_counter() - start

print(f"Temps d'entraînement CPU : {training_seconds:.2f} secondes")
print("Taille du vocabulaire :", len(baseline.named_steps["tfidf"].vocabulary_))
print("Itérations :", baseline.named_steps["classifier"].n_iter_)

## 3. Évaluer sur la validation
L'**accuracy** mesure la proportion de bonnes prédictions. Le **macro-F1** donne le même poids à chaque catégorie. Les données de test ne servent pas à régler le modèle.

In [ ]:
baseline_predictions = baseline.predict(val_df["text"])
baseline_metrics = {
    "accuracy": accuracy_score(val_df["category"], baseline_predictions),
    "macro_f1": f1_score(val_df["category"], baseline_predictions, average="macro", zero_division=0),
    "training_seconds": training_seconds,
}
print(pd.Series(baseline_metrics).round(4).to_string())
print(classification_report(val_df["category"], baseline_predictions, zero_division=0))

## 4. Examiner les erreurs

In [ ]:
validation_predictions = val_df[["text", "category"]].copy()
validation_predictions["prediction"] = baseline_predictions
errors = validation_predictions.loc[validation_predictions["category"] != validation_predictions["prediction"]]
print("Nombre d'erreurs :", len(errors))
errors.head(10)

## 5. Sauvegarder le modèle et les résultats

In [ ]:
joblib.dump(baseline, MODELS_DIR / "baseline.joblib")
(RESULTS_DIR / "baseline_metrics.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
validation_predictions.to_csv(RESULTS_DIR / "baseline_validation_predictions.csv", index=False)
print("Modèle et résultats sauvegardés. Passer au notebook 03.")